In [7]:
# -------------------------------------------
# 
# generate and save climate input files 
# 
# -------------------------------------------

# Need the following 1d arrays --------------
# 
# [ time ]: decimal year 
# [ temperature ]: degrees C
# [ soil moisture ]: mm / m
# 
import os 

import icechunk 
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import matplotlib.style as mplstyle
import numpy as np
import pandas as pd
import xarray as xr

import clim_process_helperfxns as cph

In [8]:
# select layers for mean column -------------
# for more info on soil layers, see: https://codes.ecmwf.int/grib/param-db/39 
soilwater_layers = ["volumetric_soil_water_layer_1", "volumetric_soil_water_layer_2"]
soiltemperature_layers = ["soil_temperature_level_1", "soil_temperature_level_2"]
# select the 1d variables
precip_vars = ["total_precipitation"]
temperature_vars = ["2m_temperature"]
infiltration_vars = ["sub_surface_runoff", "surface_runoff"]
potential_ET_vars = ["potential_evaporation", "evaporation"]
other_vars = ["10m_v_component_of_wind", "10m_u_component_of_wind", 
                "geopotential_at_surface", "forecast_albedo"]
var_sum_not_mean = [
    "total_precipitation", "surface_runoff",
    ]

# select time resolutions to output ---------
default_resolution = True   # native time resolution
default_res_name = "hourly" # name for the native time resolution
monthly_mean = True         # average of each month (length = 12 * nyears)
daily_mean = True           # average of each day 
daily_ltm = True           # climatological average of each day 
monthly_ltm = True          # climatological average of each month (length = 12)
annual_mean = True          # average of each year (length = nyears)
annual_ltm = True           # climatological annual average (length = 1)

# do we need to convert era5 units ----------
era5_unit_conversions_on = True

# how many times to repeat the selected years in the climate file 
nyears_repeat = 1   # repeat timeseries for all resolutions
                    # (note, longterm mean resolutions are repeated nyears_data * nyears_repeat)
roundtime_to = 5    # number of decimal places to round decimal time (5 works for hourly)

# select start and end time + sites -----------
mintime = '2020-01-01'
maxtime = '2020-12-31'
site_dict = {
    "site_1": {'xlat': 38.148, 'xlon': 360 -121.449, 'name': 'central_valley'},    # central valley
    "site_2": {'xlat': 44.764, 'xlon': 360 -93.202, 'name': 'minneapolis'},     # minneapolis
    "site_3": {'xlat': 33.584, 'xlon': 360 -83.828, 'name': 'atlanta'},     # atlanta
    "site_4": {'xlat': 42.582, 'xlon': 360 -74.029, 'name': 'albany'},     # albany
}

# where to save the data -----------------------
# (note, we'll make a subdir that is `era5_mintime_maxtime`)
savepath = "/home/tykukla/EWmodel/era5/ewmodel-ready"
save_maindir = os.path.join(savepath, f'era5_{mintime}_{maxtime}')
# subdir_rule = "combined"

# whether to save climate figure ---------------
save_clim_fig = True


In [9]:
# --- read in data
era5_dir = "era5/preprocessed_icechunk"
era5_bucket = "carbonplan-carbon-removal"

storage = icechunk.s3_storage(bucket=era5_bucket, prefix=era5_dir, from_env=True)
repo = icechunk.Repository.open(storage)
session = repo.readonly_session("main")
rtds = xr.open_zarr(session.store, consolidated=False)
rtds

<xarray.Dataset> Size: 466GB
Dimensions:                        (time: 184104, latitude: 105, longitude: 241)
Coordinates:
    level                          int64 8B ...
  * latitude                       (latitude) float32 420B 50.0 49.75 ... 24.0
  * longitude                      (longitude) float32 964B 235.0 ... 295.0
  * time                           (time) datetime64[ns] 1MB 2000-01-01 ... 2...
Data variables: (12/25)
    10m_u_component_of_wind        (time, latitude, longitude) float32 19GB dask.array<chunksize=(184104, 12, 12), meta=np.ndarray>
    10m_v_component_of_wind        (time, latitude, longitude) float32 19GB dask.array<chunksize=(184104, 12, 12), meta=np.ndarray>
    2m_temperature                 (time, latitude, longitude) float32 19GB dask.array<chunksize=(184104, 12, 12), meta=np.ndarray>
    evaporation                    (time, latitude, longitude) float32 19GB dask.array<chunksize=(184104, 12, 12), meta=np.ndarray>
    forecast_albedo                (time, latitude, longitude) float32 19GB dask.array<chunksize=(184104, 12, 12), meta=np.ndarray>
    geopotential_at_surface        (time, latitude, longitude) float32 19GB dask.array<chunksize=(184104, 12, 12), meta=np.ndarray>
    ...                             ...
    u_component_of_wind            (time, latitude, longitude) float32 19GB dask.array<chunksize=(184104, 12, 12), meta=np.ndarray>
    volumetric_soil_water_layer_2  (time, latitude, longitude) float32 19GB dask.array<chunksize=(184104, 12, 12), meta=np.ndarray>
    volumetric_soil_water_layer_3  (time, latitude, longitude) float32 19GB dask.array<chunksize=(184104, 12, 12), meta=np.ndarray>
    volumetric_soil_water_layer_1  (time, latitude, longitude) float32 19GB dask.array<chunksize=(184104, 12, 12), meta=np.ndarray>
    volumetric_soil_water_layer_4  (time, latitude, longitude) float32 19GB dask.array<chunksize=(184104, 12, 12), meta=np.ndarray>
    v_component_of_wind            (time, latitude, longitude) float32 19GB dask.array<chunksize=(184104, 12, 12), meta=np.ndarray>
Attributes:
    last_updated:           2025-09-22 01:57:27.339380+00:00
    valid_time_start:       1940-01-01
    valid_time_stop:        2025-04-30
    valid_time_stop_era5t:  2025-09-16

## [1] Create climate var dataset 

In [10]:
# --- create climate var dataset 
dsvar, nyears_data = cph.create_climvars_ds(
    rtds,
    site_dict,
    mintime, 
    maxtime,
    soilwater_layers,
    soiltemperature_layers,
    precip_vars,
    temperature_vars,
    infiltration_vars,
    potential_ET_vars,
    other_vars,
    roundtime_to,
    era5_unit_conversions_on,
)

# --- compute wind speed
dsvar['windspeed_m_s'] = np.sqrt(dsvar['10m_u_component_of_wind']**2 + dsvar['10m_v_component_of_wind']**2)

## [2] Create a dataset for each time resolution

In [11]:
ds_dict = cph.create_dsdict_across_timeResolutions(
    dsvar,
    default_res_name,
    default_resolution,
    daily_mean, 
    daily_ltm,
    monthly_mean,
    monthly_ltm,
    annual_mean,
    annual_ltm,
    var_sum_not_mean,
    roundtime_to,
)

## [3] Save resulting dict

In [12]:
# --- create a dictionary to save
if not os.path.exists(save_maindir):
    os.makedirs(save_maindir)

for key, ds in ds_dict.items():
    ds.to_netcdf(os.path.join(save_maindir, f'{key}.nc'))

In [8]:
# cph.save_all_case_climfiles_as_txt(
#     ds_dict,
#     nyears_data,
#     save_maindir,
#     inputvar_details_fn,
#     save_clim_fig,
#     subdir_rule,
# )

In [ ]:
# ----